# Job 49705064 - the leak-free N=24 task was data-bound

**Headline.** On the shared 1500-instance pool that scored the leak-free baseline at 0.8540,
tripling the training set to 240000 instances gives **1.0000** (delta +0.1460, 219 discordant
instances won and 0 lost, McNemar exact p = 2.4e-66). The cheap cell -- the same 240000
instances for 100 epochs, i.e. the *same 24M samples* as the 0.854 run -- also scores 1.0000,
so the binding constraint was instance **coverage**, not gradient steps.

**Provenance**

| | |
|---|---|
| Slurm job | 49705064, `COMPLETED`, ExitCode 0:0, elapsed 23:35:04, ended 2026-10-27T19:29:31 |
| script | `scripts/slurm/sudoku2_n24_leakfree_data.sbatch` |
| git commit at submit | `9781a31` (submitted 2026-10-26T19:43:41) |
| seed | 0 (every cell) |
| instances | N=24, degree 5, q=4, rho 0.30-0.95, removal hints, **shuffle_adj=True** |
| architecture | d_model 256, 4 heads, L=12, ff=1024, pre-norm, lr=1e-3 constant, batch 32 |
| cells | A = 240000 x 300ep, B = 240000 x 100ep, both vs finding 17's 80000 x 300ep |
| run dir | `$SCRATCH/spin_gnn/sudoku_n24_data_2026-10-26` |
| deck | `notebooks/sudoku_lfdata_2026-10-28/lfdata.pdf` (7 pages) |

The previous best on this eval is finding 17's leak-free baseline, 0.8540 [0.8352, 0.8710],
which is row 0 of every table below: it is the reference checkpoint of the same `agg_eval`
call, so the comparison is paired on identical instances rather than across two runs.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SCRATCH = os.environ["SCRATCH"]
RUN = f"{SCRATCH}/spin_gnn/sudoku_n24_data_2026-10-26"
PREV = f"{SCRATCH}/spin_gnn/sudoku_n24_leakfree_2026-10-26"
pd.set_option("display.width", 200)

CELLS = ["80k x 300ep (finding 17)", "240k x 300ep (A)", "240k x 100ep (B)"]
pool = pd.read_csv(f"{RUN}/sharedpool_n1500.csv")
pool["cell"] = CELLS   # row order = the order the checkpoints were passed, reference first
sweep = pd.concat([pd.read_csv(f"{PREV}/sweep.csv"),
                   pd.read_csv(f"{RUN}/data240k_ep300/sweep.csv"),
                   pd.read_csv(f"{RUN}/data240k_ep100/sweep.csv")], ignore_index=True)
sweep["cell"] = CELLS
depth = {"80k x 300ep (finding 17)": pd.read_csv(f"{PREV}/depth_n24_leakfree.csv"),
         "240k x 300ep (A)": pd.read_csv(f"{RUN}/depth_data240k_ep300.csv"),
         "240k x 100ep (B)": pd.read_csv(f"{RUN}/depth_data240k_ep100.csv")}
mem = pd.concat([pd.read_csv(f"{RUN}/memorization_data240k_ep300.csv"),
                 pd.read_csv(f"{RUN}/memorization_data240k_ep100.csv")], ignore_index=True)
print(len(pool), "pool rows;", len(sweep), "sweep rows;", len(mem), "memorization rows")

## 1. Metric of record: the shared n=1500 pool

`sudoku_agg_eval` builds one pool at `seed_start=50000000000` -- far above every training seed
used here (0-239999) -- and scores all three checkpoints on it, so the comparison is paired and
the test is a McNemar exact binomial on the discordant pairs.

In [ ]:
cols = ["cell", "solve_rate", "wilson_lo", "wilson_hi", "delta",
        "discordant_won", "discordant_lost", "mcnemar_exact_p", "truncated_rate"]
display(pool[cols].round(4))

scols = ["cell", "n_train_instances", "epochs", "n_params", "max_len",
         "final_val_nll", "final_val_accuracy", "mean_o1_solve_rate"]
display(sweep[scols].round(4))

# Cells A and B do not merely tie on the rate -- they agree instance by instance.
z = np.load(f"{RUN}/sharedpool_n1500_per_instance.npz", allow_pickle=True)
s = z["solved"]
print("solved per checkpoint:", s.sum(1), "of", s.shape[1])
print("A vs B discordant pairs:", int((s[1] & ~s[2]).sum()), int((~s[1] & s[2]).sum()))

### One honest asterisk on "identical architecture"

`max_len` is computed over the *training* pool, and a 240000-instance pool contains a slightly
longer trace than an 80000-instance one: 210 -> 212 tokens, so `n_params` goes 9,548,834 ->
9,549,346 (+512 = two positional-embedding rows, 0.005%), and the data cells get a 2-token
longer generation budget at eval. Neither can account for +0.146, and `truncated_rate` is
0.0000 for all three cells, so nothing was cut off at either budget.

## 2. The depth cost (pre-registration d) did not survive

`sudoku_depth_eval` bins a 600-instance pool by DD depth T at fixed *realized* density, using
the same generator stream as the baseline, so the bins are identical row by row and the
columns are comparable bin by bin.

In [ ]:
for rho in (0.60, 0.75):
    print(f"=== rho={rho}")
    t = pd.DataFrame({c: d[d.rho_requested == rho].set_index("T").solve_rate.round(4)
                      for c, d in depth.items()})
    t["n"] = depth[CELLS[0]].query("rho_requested == @rho").set_index("T").n
    display(t)

g0 = depth[CELLS[0]].query("rho_requested == 0.60")
print("baseline Pearson r(T, solve_rate) at rho=0.60:",
      round(float(np.corrcoef(g0["T"], g0.solve_rate)[0, 1]), 4))
print("cell A: every bin at 1.0000, so there is no slope to measure")
gB = depth[CELLS[2]].query("rho_requested == 0.60")
print("cell B Pearson r:", round(float(np.corrcoef(gB["T"], gB.solve_rate)[0, 1]), 4),
      "-- three bins at exactly 1.0 plus ONE missed instance in 548, so the",
      "pre-registered r test is uninformative at ceiling")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for c, col in zip(CELLS, ["#d62728", "#2ca02c", "#1f77b4"]):
    g = depth[c].query("rho_requested == 0.60").sort_values("T")
    ax.plot(g["T"], g.solve_rate, "-o", color=col, label=c)
ax.axhline(0.90, color="0.5", ls="--", lw=1)
ax.set_xlabel("DD depth T"); ax.set_ylabel("solve_rate")
ax.set_title("rho=0.60, realized density 0.583 in every bin")
ax.set_xticks([2, 3, 4, 5]); ax.legend(fontsize=9)
fig.tight_layout()

## 3. Memorization gap (pre-registration c)

`sudoku_memorization_eval` scores a recoverable prefix of the model's own training set against
a held-out draw from the same distribution. Finding 17 measured +0.1000 at 80000 instances
(0.9567 own-training vs 0.8567 held out, p=2.6e-09).

In [ ]:
mem["cell"] = ["240k x 300ep (A)", "240k x 100ep (B)"]
display(mem[["cell", "train_solve_rate", "heldout_solve_rate", "gap", "p", "n_instances"]])
print("finding 17 at 80000: train 0.9567, held-out 0.8567, gap +0.1000, p=2.6e-09")

## 4. Pre-registrations, as resolved

Written into the sbatch header before the job ran.

| | claim | verdict | evidence |
|---|---|---|---|
| (a) | Cell A clears 0.90 on the shared pool | **CONFIRMED** | 1.0000 vs 0.8540, +0.1460, p=2.4e-66 |
| (b) | Cell B also beats 0.8540, *by less* than A | **FALSIFIED** (the flagged half) | B beats it by the same +0.1460; A and B agree on all 1500 instances. The named falsifier -- "B >= A, the extra epochs do nothing" -- is what happened. |
| (c) | The memorization gap shrinks below +0.1000 | **CONFIRMED** | +0.0000 (train 1.0000 vs held-out 1.0000, p=1.0, n=600), both cells |
| (d) | The depth cost survives: r at rho=0.60 stays below -0.5 | **FALSIFIED** (in substance) | All four bins at 1.0000 in cell A. The pre-registered statistic cannot say so: cell B's r=-0.775 is one missed instance in 548 and still "passes". |

My record on data-budget predictions in this project is now 3 for 8, all in one direction:
(b) and (d) are the same underestimate again, and (d) is the expensive one, because finding
17's depth cost was the first clean evidence here that reasoning depth is what costs -- and at
N=24 it is now attributable to instance coverage instead.

## 5. What this changes

- The leak-free N=24 task is **solved**, not merely improved: 1500/1500 held out, with finding
  16's adjacency-order shortcut removed from training.
- The working budget for future leak-free runs is **240000 instances x 100 epochs**, which is
  what `scripts/slurm/sudoku2_leakfree_lstar.sbatch` has been gated on.
- Finding 17's 0.854 is demoted to "the cost of removing the leak *at 80000 instances*".
- Finding 17's N=24 depth cost is withdrawn.
- The leak-free L*(T) ladder cannot run at N=24 any more -- every cell would sit at 1.000. It
  has to move to N=48 or N=96.

## 6. What it does not change

- Nothing is established at N=48 or N=96, where traces are 2-4x longer and 240000 instances is
  3-12x the GPU time. The budget transfers as a hypothesis.
- A ceiling result cannot be ranked: this pool can no longer separate N=24 cells at all, so the
  metric of record has to move to a harder distribution (lower rho, or larger N).
- One seed.